# Tech COE Allocation Summary — Bronze Ingestion

Transforms the pivot-format Excel allocation spreadsheet from OCI Object Storage into a normalized tabular format and writes to ADW via Spark.

| | |
|---|---|
| **Source** | `/Workspace/Shared/Tech_COE_Allocation_Summary.xlsx` |
| **Sheet** | `Allocations` |
| **Target** | `arganoadw_oacuser_sh.oacuser.TECH_COE_ALLOCATIONS` |
| **Write mode** | Full overwrite each run |
| **Output schema** | DEPARTMENT, ARGANAUT, PROJECT_CODE, PROJECT_NAME, YEAR_MONTH, WEEK_START_DATE, SCHEDULED_HOURS, EXTRACTED_AT_TS, SOURCE_FILE |

In [ ]:
# ============================================================
# TECH COE ALLOCATION SUMMARY — Bronze Ingestion
# Transforms pivot-format Excel allocation spreadsheet into
# normalized tabular format and writes to ADW via Spark.
#
# Source:  /Workspace/Shared/Tech_COE_Allocation_Summary.xlsx
# Target:  arganoadw_oacuser_sh.oacuser.TECH_COE_ALLOCATIONS
# Mode:    Full overwrite each run
# ============================================================

import openpyxl
import pandas as pd
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DateType, DecimalType, TimestampType
)
from pyspark.sql.functions import current_timestamp, lit

spark = SparkSession.builder.appName("tech_coe_allocations").getOrCreate()

# ─── CONFIG ──────────────────────────────────────────────────
SOURCE_PATH  = "SOURCE_PATH = "/Volumes/cbtest_standard_catalog/default/COE_Allocations/Tech COE Allocation Summary-template.xlsx"
SHEET_NAME   = "Allocations"
TARGET_TABLE = "arganoadw_oacuser_sh.oacuser.TECH_COE_ALLOCATIONS"
# ─────────────────────────────────────────────────────────────



In [ ]:
# ── SECTION 1: Load workbook ──────────────────────────────────
# data_only=True reads cached formula values; required for openpyxl
# to see numeric hours rather than formula strings.
wb = openpyxl.load_workbook(SOURCE_PATH, data_only=True)
ws = wb[SHEET_NAME]

max_col = ws.max_column
max_row = ws.max_row
print(f"Workbook loaded: {max_row} rows × {max_col} cols")



In [ ]:
# ── SECTION 2: Build column map ───────────────────────────────
# Row 1: Year-Month labels (merged cells → sparse, needs forward-fill)
# Row 2: Start-of-week dates OR the string "Total"
# Only include columns where row 2 is a datetime — this automatically
# excludes all monthly Total columns and the grand Total column.

row1 = [ws.cell(1, c).value for c in range(1, max_col + 1)]
row2 = [ws.cell(2, c).value for c in range(1, max_col + 1)]

# Forward-fill YEAR_MONTH across merged cells in row 1
year_month_fwd = {}
current_ym = None
for i, val in enumerate(row1):
    if val is not None and val not in ("Year - Month", "Total"):
        current_ym = val
    year_month_fwd[i] = current_ym

# Build week column index: only datetime-valued cells in row 2
week_columns = []
for i, val in enumerate(row2):
    if isinstance(val, datetime):
        week_columns.append({
            "col_idx":        i + 1,                  # openpyxl is 1-based
            "year_month":     year_month_fwd[i],
            "week_start_date": val.date()
        })

months = sorted(set(w["year_month"] for w in week_columns))
print(f"Week columns: {len(week_columns)} weeks across {len(months)} months")
print(f"Months: {months}")



In [ ]:
# ── SECTION 3: Parse hierarchical row structure ───────────────
# The spreadsheet uses 3 row types distinguished by which columns
# are populated:
#
#   Dept subtotal row  → col A = dept name,  col B = "Total", col C = None
#   Person subtotal    → col A = None,        col B = full name, col C = "Total"
#   Project detail     → col A = None,        col B = None,      col C = project code/name
#
# We forward-fill DEPARTMENT and ARGANAUT as we descend the rows.
# Subtotal rows are skipped; only project detail rows emit records.
# Zero-hour weeks are retained as 0.0 (not dropped).

records = []
current_dept     = None
current_arganaut = None
skipped_rows     = 0
detail_rows      = 0

for r in range(4, max_row + 1):          # Row 1-3 are headers
    col_a = ws.cell(r, 1).value
    col_b = ws.cell(r, 2).value
    col_c = ws.cell(r, 3).value

    # ── Department subtotal row ───────────────────────────────
    if col_a is not None and col_b == "Total":
        current_dept = col_a
        skipped_rows += 1
        continue

    # ── Person subtotal row ───────────────────────────────────
    if col_b is not None and col_b != "Total" and col_c == "Total":
        current_arganaut = col_b
        skipped_rows += 1
        continue

    # ── Project detail row ────────────────────────────────────
    if col_c is not None and col_c != "Total":
        detail_rows += 1

        # Split "ARGU00666.01 - Project Name" on first " - " only
        raw_project = str(col_c).strip()
        if " - " in raw_project:
            project_code, project_name = raw_project.split(" - ", 1)
            project_code  = project_code.strip()
            project_name  = project_name.strip()
        else:
            project_code  = raw_project
            project_name  = None

        for wk in week_columns:
            raw_hours = ws.cell(r, wk["col_idx"]).value
            hours     = round(float(raw_hours), 4) if raw_hours is not None else 0.0

            records.append({
                "DEPARTMENT":       current_dept,
                "ARGANAUT":         current_arganaut,
                "PROJECT_CODE":     project_code,
                "PROJECT_NAME":     project_name,
                "YEAR_MONTH":       wk["year_month"],
                "WEEK_START_DATE":  wk["week_start_date"],
                "SCHEDULED_HOURS":  hours,
            })
        continue

    # ── Blank / padding row ───────────────────────────────────
    skipped_rows += 1

print(f"Detail rows parsed:   {detail_rows}")
print(f"Subtotal/blank rows:  {skipped_rows}")
print(f"Records generated:    {len(records)}")
print(f"  └─ With hours > 0:  {sum(1 for r in records if r['SCHEDULED_HOURS'] > 0)}")
print(f"  └─ With hours = 0:  {sum(1 for r in records if r['SCHEDULED_HOURS'] == 0)}")



In [ ]:
# ── SECTION 4: Build Spark DataFrame ─────────────────────────
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DateType, DoubleType, TimestampType  # ← DoubleType replaces DecimalType
)

schema = StructType([
    StructField("DEPARTMENT",       StringType(),  True),
    StructField("ARGANAUT",         StringType(),  True),
    StructField("PROJECT_CODE",     StringType(),  True),
    StructField("PROJECT_NAME",     StringType(),  True),
    StructField("YEAR_MONTH",       StringType(),  True),
    StructField("WEEK_START_DATE",  DateType(),    True),
    StructField("SCHEDULED_HOURS",  DoubleType(),  True),  # ← was DecimalType(10,4)
])

pdf = pd.DataFrame(records)
pdf["WEEK_START_DATE"] = pd.to_datetime(pdf["WEEK_START_DATE"])
pdf["SCHEDULED_HOURS"] = pdf["SCHEDULED_HOURS"].astype(float)

df = (spark.createDataFrame(pdf, schema=schema)
          .withColumn("EXTRACTED_AT_TS", current_timestamp())
          .withColumn("SOURCE_FILE",     lit(SOURCE_PATH)))

print(f"\nDataFrame: {df.count()} rows, {len(df.columns)} columns")
df.printSchema()

print("\n── Preview (non-zero hours) ──────────────────────────────")
df.filter("SCHEDULED_HOURS > 0").show(15, truncate=False)



In [ ]:
# ── SECTION 5: Write to ADW ───────────────────────────────────
# External catalog rules:
#   • No .format("delta") — ADW does not support Delta format
#   • No CREATE SCHEMA — returns 502 from AIDP Metastore
#   • Use saveAsTable with 3-part name, overwrite + overwriteSchema

print(f"\nWriting to {TARGET_TABLE} ...")

(df.write
   .mode("overwrite")
   .option("overwriteSchema", "true")
   .saveAsTable(TARGET_TABLE))

print(f"✅ Write complete.")



In [ ]:
# ── SECTION 6: Verify ─────────────────────────────────────────
spark.sql(f"SELECT COUNT(*) AS TOTAL_ROWS FROM {TARGET_TABLE}").show()

spark.sql(f"""
    SELECT
        DEPARTMENT,
        ARGANAUT,
        YEAR_MONTH,
        COUNT(*)                        AS WEEK_PROJECT_ROWS,
        ROUND(SUM(SCHEDULED_HOURS), 2)  AS TOTAL_HOURS
    FROM {TARGET_TABLE}
    WHERE SCHEDULED_HOURS > 0
    GROUP BY DEPARTMENT, ARGANAUT, YEAR_MONTH
    ORDER BY DEPARTMENT, ARGANAUT, YEAR_MONTH
""").show(50, truncate=False)
